 ## *Download all CV files (Google Drive links) from the Excel shee*

In [2]:
'''# ============================================================

# Run this in Google Colab
# ============================================================

# 1. Install/upgrade needed packages (requests is left alone -- Colab
#    already ships a compatible version, and upgrading it can trigger
#    dependency-conflict warnings with google-colab's own tools)
!pip install -q --upgrade openpyxl

import re
import os
import openpyxl
import requests

# ------------------------------------------------------------
# 2. Upload the Excel file (skip this cell if you already
#    uploaded it to Colab's file browser on the left)
# ------------------------------------------------------------
from google.colab import files
uploaded = files.upload()   # choose your .xlsx file when prompted
EXCEL_PATH = list(uploaded.keys())[0]

# ------------------------------------------------------------
# 3. Create the output folder named "CVs" using an ABSOLUTE path
#    (so we always know exactly where files land)
# ------------------------------------------------------------
OUTPUT_FOLDER = "/content/CVs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ------------------------------------------------------------
# 4. Read the Excel file and collect (Name, Drive link) pairs
#    Sheet columns: Full Name | University | CV | standard_univ
# ------------------------------------------------------------
wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb.worksheets[0]  # first sheet

rows = list(ws.iter_rows(min_row=2, values_only=True))  # skip header row

entries = []
for row in rows:
    name = row[0]
    link = row[2]  # "CV" column
    if name and link and "drive.google.com" in str(link):
        entries.append((str(name).strip(), str(link).strip()))

print(f"Found {len(entries)} CV links to download.")

# ------------------------------------------------------------
# 5. Extract the Google Drive file ID from any link format
# ------------------------------------------------------------
def extract_file_id(url: str):
    patterns = [
        r"/d/([a-zA-Z0-9_-]+)",
        r"id=([a-zA-Z0-9_-]+)",
    ]
    for pat in patterns:
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return None

def safe_filename(name: str) -> str:
    return re.sub(r'[\\/*?:"<>|]', "", name).strip()

# ------------------------------------------------------------
# 6. Robust download function using requests directly.
#    Handles Google's "can't scan large file for viruses"
#    confirmation page, which is what silently breaks gdown
#    in a lot of Colab sessions.
# ------------------------------------------------------------
def download_drive_file(file_id: str, dest_path_no_ext: str, session: requests.Session):
    URL = "https://drive.google.com/uc?export=download"
    response = session.get(URL, params={"id": file_id}, stream=True)

    # Look for a confirm token (appears for large files)
    token = None
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            token = value
            break

    if token is None:
        # Some responses embed the confirm token in the HTML instead of cookies
        if "text/html" in response.headers.get("Content-Type", ""):
            match = re.search(r"confirm=([0-9A-Za-z_-]+)", response.text)
            if match:
                token = match.group(1)

    if token:
        response = session.get(URL, params={"id": file_id, "confirm": token}, stream=True)

    content_type = response.headers.get("Content-Type", "")
    content_disp = response.headers.get("Content-Disposition", "")

    # Figure out a reasonable extension
    ext = ".pdf"  # sensible default for CVs
    m = re.search(r'filename="?([^";]+)"?', content_disp)
    if m:
        original_name = m.group(1)
        if "." in original_name:
            ext = "." + original_name.rsplit(".", 1)[-1]

    dest_path = dest_path_no_ext + ext

    if "text/html" in content_type:
        # We got an HTML page, not the actual file -> download failed
        return None

    with open(dest_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=32768):
            if chunk:
                f.write(chunk)

    # Sanity check: make sure the file isn't suspiciously tiny (likely an error page)
    if os.path.getsize(dest_path) < 1024:
        with open(dest_path, "rb") as f:
            head = f.read(200).lower()
        if b"<html" in head or b"<!doctype" in head:
            os.remove(dest_path)
            return None

    return dest_path

# ------------------------------------------------------------
# 7. Download each file into the CVs folder
# ------------------------------------------------------------
failed = []
succeeded = []
session = requests.Session()

for i, (name, link) in enumerate(entries, start=1):
    file_id = extract_file_id(link)
    if not file_id:
        print(f"[{i}/{len(entries)}] Could not parse file ID for {name}: {link}")
        failed.append((name, link))
        continue

    dest_prefix = os.path.join(OUTPUT_FOLDER, safe_filename(name))
    try:
        result_path = download_drive_file(file_id, dest_prefix, session)
        if result_path:
            size_kb = os.path.getsize(result_path) / 1024
            print(f"[{i}/{len(entries)}] OK: {os.path.basename(result_path)} ({size_kb:.1f} KB)")
            succeeded.append((name, result_path))
        else:
            print(f"[{i}/{len(entries)}] FAILED (file not public or blocked) for {name}")
            failed.append((name, link))
    except Exception as e:
        print(f"[{i}/{len(entries)}] FAILED for {name}: {e}")
        failed.append((name, link))

# ------------------------------------------------------------
# 8. VERIFY what's actually on disk (don't trust the log alone)
# ------------------------------------------------------------
actual_files = os.listdir(OUTPUT_FOLDER)
print("\n" + "=" * 60)
print(f"Verified: {len(actual_files)} file(s) physically present in {OUTPUT_FOLDER}")
print("=" * 60)
for f in actual_files:
    full = os.path.join(OUTPUT_FOLDER, f)
    print(f" - {f}  ({os.path.getsize(full)/1024:.1f} KB)")

if failed:
    print(f"\n{len(failed)} file(s) failed to download (likely not shared as 'Anyone with the link'):")
    for name, link in failed:
        print(f" - {name}: {link}")

# ------------------------------------------------------------
# 9. Zip the CVs folder and download it to your computer
#    (Only run this after confirming the file count above looks right.
#     Also remember: refresh the Colab file browser panel on the left
#     with the circular-arrow icon -- it does NOT auto-refresh.)
# ------------------------------------------------------------
!cd /content && zip -r CVs.zip CVs
files.download("/content/CVs.zip")'''

'# ============================================================\n\n# Run this in Google Colab\n# ============================================================\n\n# 1. Install/upgrade needed packages (requests is left alone -- Colab\n#    already ships a compatible version, and upgrading it can trigger\n#    dependency-conflict warnings with google-colab\'s own tools)\n!pip install -q --upgrade openpyxl\n\nimport re\nimport os\nimport openpyxl\nimport requests\n\n# ------------------------------------------------------------\n# 2. Upload the Excel file (skip this cell if you already\n#    uploaded it to Colab\'s file browser on the left)\n# ------------------------------------------------------------\nfrom google.colab import files\nuploaded = files.upload()   # choose your .xlsx file when prompted\nEXCEL_PATH = list(uploaded.keys())[0]\n\n# ------------------------------------------------------------\n# 3. Create the output folder named "CVs" using an ABSOLUTE path\n#    (so we always k

In [ ]:
""" from google.colab import drive
drive.mount('/content/drive')

cv_folder = '/content/drive/MyDrive/CVs'  """

Mounted at /content/drive


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
import shutil

cv_folder = '/content/CVs'
os.makedirs(cv_folder, exist_ok=True)

zip_path = '/content/CVs.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    for file in zip_ref.namelist():

        # Ignore folders
        if file.endswith('/'):
            continue

        # Get only the filename
        filename = os.path.basename(file)

        # Skip hidden/system files
        if filename.startswith('.') or not filename:
            continue

        # Extract directly into /content/CVs
        destination = os.path.join(cv_folder, filename)

        with zip_ref.open(file) as source, open(destination, 'wb') as target:
            shutil.copyfileobj(source, target)

print("All CVs extracted successfully!")

# Count CV files
cv_files = [
    f for f in os.listdir(cv_folder)
    if f.lower().endswith(('.pdf', '.docx', '.doc'))
]

print(f"Total CVs found: {len(cv_files)}")

for cv in cv_files:
    print(cv)

Mounted at /content/drive
All CVs extracted successfully!
Total CVs found: 98
Hamza Ali Bakr.docx
Ahmed Osama.pdf
Moataz Badawy.pdf
Hassan Hossam eldain hassan.docx
Abdallah Yahia Mansour Mohammed.pdf
Bassel Hesham Farouk.pdf
Abdelrahman Amr.pdf
Mohamed Hisham Wafa.docx
Youssef Mohamed Kenawy.pdf
Ziad Basem El Gamal.pdf
Loai Farrag.docx
Yassin Mohamed Amer.docx
Omar tamer momtaz.pdf
Omar Ahmed Abdelraouf.pdf
Sally Ahmed Mostafa Mohamed.pdf
Marwan sherif saleh.pdf
Habiba Ahmed Aboelenen.pdf
Noor Omar Mohamed Elsamahy.docx
Karim Mohamed Abdelaziz Mokadem.pdf
Maria Ashraf.pdf
Malak Ahmed.pdf
Alaa Karam Ahmed Mohamed Salama.pdf
Abdelrahaman ehab.docx
Omar hazem algaradawy.pdf
Andrew Refaat.pdf
Moustafa Hani Moustafa Mansour.pdf
sylvia faris wardkhan.pdf
Khaled Ehab Abdelfattah.pdf
Ahmed Maged Mounir Mohammed.pdf
Natalia Ehab Habib.pdf
Mahmoud Samir Mahmoud Moustafa.pdf
Yehia Mohamed Mohamed Alsaeed.pdf
Hassan Mohammed Wael aburezq.pdf
Yassin Saleh Ahmed.pdf
Yousef Elbrolosy.pdf
nourhan moh

In [4]:
!pip install pymupdf python-docx pytesseract sentence-transformers scikit-learn pandas pillow
!apt-get install -y tesseract-ocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.9 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 24 not upgraded.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import fitz  # PyMuPDF
import docx
import pytesseract
from PIL import Image
import os

def extract_text(filepath):
    ext = filepath.lower().split('.')[-1]
    text = ""

    if ext == 'pdf':
        doc = fitz.open(filepath)
        for page in doc:
            text += page.get_text()
        doc.close()

    elif ext == 'docx':
        d = docx.Document(filepath)
        text = "\n".join([p.text for p in d.paragraphs])

    elif ext == 'png':
        img = Image.open(filepath)
        text = pytesseract.image_to_string(img)

    return text.strip()

In [7]:
cv_texts = {}

for filename in os.listdir(cv_folder):
    filepath = os.path.join(cv_folder, filename)
    try:
        text = extract_text(filepath)
        if text:  # skip empty extractions
            cv_texts[filename] = text
    except Exception as e:
        print(f"Failed to read {filename}: {e}")

print(f"Successfully extracted {len(cv_texts)} CVs")

Successfully extracted 97 CVs


# AI CV Job Matching System

The original Google Drive CV downloader is preserved above. The matching pipeline below is designed for the AI Internship criteria and supports 100+ CVs.

It combines rule-based skill matching, semantic similarity, evidence scoring, and an AI/domain relevance gate.

## 1. Install Dependencies

In [8]:
!pip install -q pymupdf python-docx sentence-transformers scikit-learn pandas openpyxl tqdm

## 2. Import Libraries and Configure the Pipeline

In [9]:
import os
import re
import numpy as np
import pandas as pd
import fitz
import docx
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

CV_FOLDER = "/content/CVs"

# Final-score weights. Easy to change.
KEYWORD_WEIGHT = 0.35
SEMANTIC_WEIGHT = 0.35
EVIDENCE_WEIGHT = 0.30

# Unrelated CVs below this AI/technical relevance level are rejected.
AI_RELEVANCE_THRESHOLD = 0.20

TOP_N = 20

def decision_from_score(score):
    if score == 0:
        return "REJECT"
    if score >= 90:
        return "EXCELLENT MATCH"
    if score >= 75:
        return "STRONG MATCH"
    if score >= 60:
        return "GOOD MATCH"
    if score >= 10:
        return "WEAK MATCH"
    return "VERY WEAK MATCH"

print("Configuration loaded.")

Configuration loaded.


## 3. Extract and Clean CV Text

In [10]:
def extract_pdf_text(filepath):
    parts = []
    with fitz.open(filepath) as pdf:
        for page in pdf:
            parts.append(page.get_text())
    return "\n".join(parts).strip()

def extract_docx_text(filepath):
    document = docx.Document(filepath)
    parts = [p.text for p in document.paragraphs if p.text.strip()]
    for table in document.tables:
        for row in table.rows:
            parts.append(" ".join(cell.text for cell in row.cells if cell.text.strip()))
    return "\n".join(parts).strip()

def extract_txt_text(filepath):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

def extract_text(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".pdf":
        return extract_pdf_text(filepath)
    if ext == ".docx":
        return extract_docx_text(filepath)
    if ext == ".txt":
        return extract_txt_text(filepath)
    raise ValueError(f"Unsupported file type: {ext}")

def clean_text(text):
    text = str(text).lower()
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

cv_records = []

if not os.path.isdir(CV_FOLDER):
    raise FileNotFoundError(f"CV folder not found: {CV_FOLDER}")

for filename in sorted(os.listdir(CV_FOLDER)):
    filepath = os.path.join(CV_FOLDER, filename)
    if not os.path.isfile(filepath):
        continue

    ext = os.path.splitext(filename)[1].lower()
    if ext not in {".pdf", ".docx", ".txt"}:
        continue

    try:
        raw_text = extract_text(filepath)
        candidate_name = os.path.splitext(filename)[0]

        if not raw_text.strip():
            status = "EMPTY"
            raw_text = ""
        else:
            status = "OK"

        cv_records.append({
            "filename": filename,
            "filepath": filepath,
            "candidate_name": candidate_name,
            "full_text": raw_text,
            "clean_text": clean_text(raw_text),
            "file_type": ext.replace(".", "").upper(),
            "processing_status": status
        })

    except Exception as e:
        cv_records.append({
            "filename": filename,
            "filepath": filepath,
            "candidate_name": os.path.splitext(filename)[0],
            "full_text": "",
            "clean_text": "",
            "file_type": ext.replace(".", "").upper(),
            "processing_status": f"ERROR: {str(e)[:150]}"
        })

cv_df = pd.DataFrame(cv_records)

print(f"Found {len(cv_df)} supported CV file(s).")
if len(cv_df):
    print(cv_df["processing_status"].value_counts(dropna=False))

Found 98 supported CV file(s).
processing_status
OK       97
EMPTY     1
Name: count, dtype: int64


## 4. Define AI Internship Requirements

In [11]:
JOB_REQUIREMENTS = {
    "ml_fundamentals": {
        "name": "ML Fundamentals",
        "weight": 0.20,
        "description": "Strong understanding of Machine Learning fundamentals.",
        "keywords": [
            "machine learning", "ml", "supervised learning",
            "unsupervised learning", "classification", "regression",
            "clustering", "feature engineering", "model selection"
        ]
    },
    "model_training_evaluation": {
        "name": "Model Training / Evaluation / Optimization",
        "weight": 0.20,
        "description": "Experience training, evaluating, and optimizing Machine Learning models.",
        "keywords": [
            "model training", "training models", "model evaluation",
            "evaluation", "optimization", "hyperparameter tuning",
            "cross validation", "cross-validation", "accuracy",
            "precision", "recall", "f1 score", "f1-score", "roc auc"
        ]
    },
    "ai_projects": {
        "name": "Hands-on AI Projects",
        "weight": 0.20,
        "description": "Hands-on AI projects such as Chatbots, NLP, or Computer Vision.",
        "keywords": [
            "artificial intelligence", "ai project", "ai projects",
            "chatbot", "chatbots", "nlp", "natural language processing",
            "sentiment analysis", "text classification", "language model",
            "computer vision", "opencv", "image classification",
            "object detection", "image processing"
        ]
    },
    "backend": {
        "name": "Backend Development",
        "weight": 0.10,
        "description": "Good understanding of Backend Development concepts.",
        "keywords": [
            "backend", "back-end", "server-side", "backend development",
            "server development"
        ]
    },
    "rest_api": {
        "name": "REST APIs / FastAPI / Flask",
        "weight": 0.15,
        "description": "Experience working with REST APIs, preferably FastAPI or Flask.",
        "keywords": [
            "rest api", "restful api", "api development",
            "fastapi", "fast api", "flask", "api"
        ]
    },
    "deployment": {
        "name": "AI Model Integration / Deployment",
        "weight": 0.15,
        "description": "Experience integrating and deploying AI models through APIs in real-world applications.",
        "keywords": [
            "model deployment", "ml deployment", "ai deployment",
            "model serving", "deployment", "deployed model",
            "model integration", "ai integration", "api integration",
            "production", "docker"
        ]
    }
}

weight_sum = sum(x["weight"] for x in JOB_REQUIREMENTS.values())
for x in JOB_REQUIREMENTS.values():
    x["normalized_weight"] = x["weight"] / weight_sum

JOB_DESCRIPTION = "\n".join(
    f"{x['name']}: {x['description']}"
    for x in JOB_REQUIREMENTS.values()
)

print(JOB_DESCRIPTION)

ML Fundamentals: Strong understanding of Machine Learning fundamentals.
Model Training / Evaluation / Optimization: Experience training, evaluating, and optimizing Machine Learning models.
Hands-on AI Projects: Hands-on AI projects such as Chatbots, NLP, or Computer Vision.
Backend Development: Good understanding of Backend Development concepts.
REST APIs / FastAPI / Flask: Experience working with REST APIs, preferably FastAPI or Flask.
AI Model Integration / Deployment: Experience integrating and deploying AI models through APIs in real-world applications.


## 5. AI / Technical Relevance Gate

In [12]:
AI_RELEVANCE_GROUPS = {
    "ai_ml": [
        "artificial intelligence", "machine learning", "deep learning",
        "neural network", "neural networks", "nlp",
        "natural language processing", "computer vision",
        "model training", "model evaluation", "scikit-learn",
        "tensorflow", "pytorch", "xgboost"
    ],
    "ai_projects": [
        "chatbot", "sentiment analysis", "object detection",
        "image classification", "text classification",
        "recommendation system", "recommendation engine",
        "generative ai", "genai", "llm"
    ],
    "technical_backend": [
        "python", "fastapi", "flask", "rest api",
        "restful api", "backend", "backend development",
        "api development", "model deployment", "model serving"
    ]
}

def count_group_hits(text, phrases):
    return sum(
        1 for phrase in phrases
        if re.search(r"\b" + re.escape(phrase.lower()) + r"\b", text)
    )

def calculate_ai_relevance(text):
    """
    AI relevance = actual match against the AI internship requirements.

    Each requirement is weighted according to JOB_REQUIREMENTS.
    This is the score that should be used for ranking candidates.
    """

    if not text:
        return 0.0

    requirement_scores = {}

    for req_key, req in JOB_REQUIREMENTS.items():

        keyword_score, matched_keywords = calculate_keyword_score(
            text,
            req
        )

        # If there is no actual keyword evidence,
        # don't allow semantic similarity alone to make the
        # candidate look highly relevant.
        if not matched_keywords:
            requirement_scores[req_key] = 0.0
            continue

        requirement_scores[req_key] = keyword_score

    # Weighted AI job relevance
    ai_relevance = sum(
        requirement_scores[key] *
        JOB_REQUIREMENTS[key]["normalized_weight"]
        for key in JOB_REQUIREMENTS
    )

    return float(min(max(ai_relevance, 0.0), 1.0))

BASE_ON_FIRST_MATCH = 0.6
BONUS_PER_EXTRA_MATCH = 0.15
MAX_EXTRA_MATCHES_COUNTED = int((1.0 - BASE_ON_FIRST_MATCH) / BONUS_PER_EXTRA_MATCH)

def calculate_keyword_score(text, requirement):
    if not text:
        return 0.0, []
    text = clean_text(text)
    matched = []
    for kw in requirement["keywords"]:
        pattern = r"\b" + re.escape(kw.lower()) + r"\b"
        if re.search(pattern, text):
            matched.append(kw)
    matched = list(dict.fromkeys(matched))  # dedupe, keep order
    if not matched:
        return 0.0, matched
    extra = min(len(matched) - 1, MAX_EXTRA_MATCHES_COUNTED)
    score = min(BASE_ON_FIRST_MATCH + extra * BONUS_PER_EXTRA_MATCH, 1.0)
    return float(score), matched



def evidence_snippet(text, matched_terms, max_chars=300):
    if not text or not matched_terms:
        return "Not Found"

    text_clean = re.sub(r"\s+", " ", text).strip()

    for term in matched_terms:
        idx = text_clean.lower().find(term.lower())
        if idx >= 0:
            start = max(0, idx - 100)
            end = min(len(text_clean), idx + max_chars)
            return text_clean[start:end]

    return "Not Found"

## 6. Load Sentence Transformer Model

In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")

requirement_items = list(JOB_REQUIREMENTS.items())
requirement_texts = [x["description"] for _, x in requirement_items]

requirement_embeddings = model.encode(
    requirement_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Semantic model loaded once.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Semantic model loaded once.


## 7. Score Every Candidate

In [14]:
valid_mask = (
    cv_df["processing_status"].eq("OK") &
    cv_df["clean_text"].str.len().gt(0)
)

valid_indices = cv_df.index[valid_mask].tolist()
candidate_texts = cv_df.loc[valid_indices, "clean_text"].tolist()

if candidate_texts:
    candidate_embeddings = model.encode(
        candidate_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=True
    )
else:
    candidate_embeddings = np.empty((0, requirement_embeddings.shape[1]))

results = []

for pos, idx in enumerate(tqdm(valid_indices, desc="Scoring CVs")):
    row = cv_df.loc[idx]
    text = row["clean_text"]

    ai_relevance = calculate_ai_relevance(text)

    keyword_scores = {}
    semantic_scores = {}
    evidence_scores = {}
    matched_skills = []
    missing_skills = []
    evidence = {}

    for req_pos, (req_key, req) in enumerate(requirement_items):
        k_score, matched = calculate_keyword_score(text, req)

        semantic = float(
            cosine_similarity(
                [candidate_embeddings[pos]],
                [requirement_embeddings[req_pos]]
            )[0][0]
        )
        semantic = max(0.0, min(1.0, semantic))

        evidence_score = min(
            1.0,
            0.70 * k_score + 0.30 * (1.0 if matched else 0.0)
        )

        keyword_scores[req_key] = k_score
        semantic_scores[req_key] = semantic
        evidence_scores[req_key] = evidence_score

        if matched:
            matched_skills.extend(matched)
            evidence[req["name"]] = evidence_snippet(text, matched)
        else:
            missing_skills.append(req["name"])
            evidence[req["name"]] = "Not Found"

    keyword_total = sum(
        keyword_scores[k] * JOB_REQUIREMENTS[k]["normalized_weight"]
        for k in JOB_REQUIREMENTS
    )
    semantic_total = sum(
        semantic_scores[k] * JOB_REQUIREMENTS[k]["normalized_weight"]
        for k in JOB_REQUIREMENTS
    )
    evidence_total = sum(
        evidence_scores[k] * JOB_REQUIREMENTS[k]["normalized_weight"]
        for k in JOB_REQUIREMENTS
    )

    base_score = (
        KEYWORD_WEIGHT * keyword_total +
        SEMANTIC_WEIGHT * semantic_total +
        EVIDENCE_WEIGHT * evidence_total
    )

    if ai_relevance < AI_RELEVANCE_THRESHOLD:
        final_score = 0.0
        decision = "REJECT"
    else:
        final_score = round(base_score * 100, 2)
        decision = decision_from_score(final_score)

    result = {
        "candidate_name": row["candidate_name"],
        "filename": row["filename"],
        "file_type": row["file_type"],
        "processing_status": row["processing_status"],
        "ai_relevance_percent": round(ai_relevance * 100, 2),
        "keyword_percent": round(keyword_total * 100, 2),
        "semantic_percent": round(semantic_total * 100, 2),
        "evidence_percent": round(evidence_total * 100, 2),
        "final_match_percent": final_score,
        "decision": decision,
        "matched_skills": ", ".join(sorted(set(matched_skills))),
        "missing_skills": ", ".join(missing_skills),
        "strong_evidence": " | ".join(
            f"{k}: {v}" for k, v in evidence.items() if v != "Not Found"
        ) or "Not Found"
    }

    for req_key, req in JOB_REQUIREMENTS.items():
        result[f"{req['name']} - Keyword"] = round(
            keyword_scores[req_key] * 100, 2
        )
        result[f"{req['name']} - Semantic"] = round(
            semantic_scores[req_key] * 100, 2
        )
        result[f"{req['name']} - Evidence"] = round(
            evidence_scores[req_key] * 100, 2
        )

    results.append(result)

# Preserve empty/error CVs in the final report.
for idx in cv_df.index[~valid_mask]:
    row = cv_df.loc[idx]
    result = {
        "candidate_name": row["candidate_name"],
        "filename": row["filename"],
        "file_type": row["file_type"],
        "processing_status": row["processing_status"],
        "ai_relevance_percent": 0.0,
        "keyword_percent": 0.0,
        "semantic_percent": 0.0,
        "evidence_percent": 0.0,
        "final_match_percent": 0.0,
        "decision": "REJECT - FILE ERROR/EMPTY",
        "matched_skills": "",
        "missing_skills": ", ".join(x["name"] for x in JOB_REQUIREMENTS.values()),
        "strong_evidence": "Not Available"
    }

    for req in JOB_REQUIREMENTS.values():
        result[f"{req['name']} - Keyword"] = 0.0
        result[f"{req['name']} - Semantic"] = 0.0
        result[f"{req['name']} - Evidence"] = 0.0

    results.append(result)

results_df = pd.DataFrame(results)
print(f"Scored {len(results_df)} CVs.")

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Scoring CVs:   0%|          | 0/97 [00:00<?, ?it/s]

Scored 98 CVs.


## 8. Rank Candidates

In [15]:
results_df = results_df.sort_values(
    by=["final_match_percent", "ai_relevance_percent"],
    ascending=False
).reset_index(drop=True)

results_df.insert(0, "rank", np.arange(1, len(results_df) + 1))

preview_cols = [
    "rank", "candidate_name", "filename",
    "ai_relevance_percent", "final_match_percent", "decision"
]

display(results_df[preview_cols].head(TOP_N))

,rank,candidate_name,filename,ai_relevance_percent,final_match_percent,decision
0,1,Omar Mohamed,Omar Mohamed.pdf,81.75,68.97,GOOD MATCH
1,2,Salma Taha Abdelalem Eloraby,Salma Taha Abdelalem Eloraby.pdf,82.50,68.12,GOOD MATCH
2,3,Youssef Mohamed Kenawy,Youssef Mohamed Kenawy.pdf,84.75,67.49,GOOD MATCH
3,4,Hamza Ali Bakr,Hamza Ali Bakr.docx,79.50,65.25,GOOD MATCH
4,5,Marwa Ahmed Yehia Ibrahem,Marwa Ahmed Yehia Ibrahem.pdf,73.50,63.92,GOOD MATCH
5,6,Hazem Ismail Ismail Mostafa Zidan,Hazem Ismail Ismail Mostafa Zidan.pdf,77.25,63.59,GOOD MATCH
6,7,Marwan Montasser Morad Hassan,Marwan Montasser Morad Hassan.pdf,76.50,63.54,GOOD MATCH
7,8,Jomana Gehad Mohamed,Jomana Gehad Mohamed.docx,74.25,62.66,GOOD MATCH
8,9,Loai Farrag,Loai Farrag.docx,80.25,62.05,GOOD MATCH
9,10,Abdelrahman Mohamed Elkady,Abdelrahman Mohamed Elkady.pdf,75.75,61.43,GOOD MATCH


## 9. Top Candidates and Rejected Candidates

In [16]:
for n in [5, 10, TOP_N]:
    n = min(n, len(results_df))
    print(f"\nTOP {n}")
    display(results_df[preview_cols].head(n))

rejected_df = results_df[
    results_df["decision"].str.startswith("REJECT")
].copy()

print("\nREJECTED CANDIDATES")
display(rejected_df[preview_cols])


TOP 5


,rank,candidate_name,filename,ai_relevance_percent,final_match_percent,decision
0,1,Omar Mohamed,Omar Mohamed.pdf,81.75,68.97,GOOD MATCH
1,2,Salma Taha Abdelalem Eloraby,Salma Taha Abdelalem Eloraby.pdf,82.50,68.12,GOOD MATCH
2,3,Youssef Mohamed Kenawy,Youssef Mohamed Kenawy.pdf,84.75,67.49,GOOD MATCH
3,4,Hamza Ali Bakr,Hamza Ali Bakr.docx,79.50,65.25,GOOD MATCH
4,5,Marwa Ahmed Yehia Ibrahem,Marwa Ahmed Yehia Ibrahem.pdf,73.50,63.92,GOOD MATCH



TOP 10


,rank,candidate_name,filename,ai_relevance_percent,final_match_percent,decision
0,1,Omar Mohamed,Omar Mohamed.pdf,81.75,68.97,GOOD MATCH
1,2,Salma Taha Abdelalem Eloraby,Salma Taha Abdelalem Eloraby.pdf,82.50,68.12,GOOD MATCH
2,3,Youssef Mohamed Kenawy,Youssef Mohamed Kenawy.pdf,84.75,67.49,GOOD MATCH
3,4,Hamza Ali Bakr,Hamza Ali Bakr.docx,79.50,65.25,GOOD MATCH
4,5,Marwa Ahmed Yehia Ibrahem,Marwa Ahmed Yehia Ibrahem.pdf,73.50,63.92,GOOD MATCH
5,6,Hazem Ismail Ismail Mostafa Zidan,Hazem Ismail Ismail Mostafa Zidan.pdf,77.25,63.59,GOOD MATCH
6,7,Marwan Montasser Morad Hassan,Marwan Montasser Morad Hassan.pdf,76.50,63.54,GOOD MATCH
7,8,Jomana Gehad Mohamed,Jomana Gehad Mohamed.docx,74.25,62.66,GOOD MATCH
8,9,Loai Farrag,Loai Farrag.docx,80.25,62.05,GOOD MATCH
9,10,Abdelrahman Mohamed Elkady,Abdelrahman Mohamed Elkady.pdf,75.75,61.43,GOOD MATCH



TOP 20


,rank,candidate_name,filename,ai_relevance_percent,final_match_percent,decision
0,1,Omar Mohamed,Omar Mohamed.pdf,81.75,68.97,GOOD MATCH
1,2,Salma Taha Abdelalem Eloraby,Salma Taha Abdelalem Eloraby.pdf,82.50,68.12,GOOD MATCH
2,3,Youssef Mohamed Kenawy,Youssef Mohamed Kenawy.pdf,84.75,67.49,GOOD MATCH
3,4,Hamza Ali Bakr,Hamza Ali Bakr.docx,79.50,65.25,GOOD MATCH
4,5,Marwa Ahmed Yehia Ibrahem,Marwa Ahmed Yehia Ibrahem.pdf,73.50,63.92,GOOD MATCH
5,6,Hazem Ismail Ismail Mostafa Zidan,Hazem Ismail Ismail Mostafa Zidan.pdf,77.25,63.59,GOOD MATCH
6,7,Marwan Montasser Morad Hassan,Marwan Montasser Morad Hassan.pdf,76.50,63.54,GOOD MATCH
7,8,Jomana Gehad Mohamed,Jomana Gehad Mohamed.docx,74.25,62.66,GOOD MATCH
8,9,Loai Farrag,Loai Farrag.docx,80.25,62.05,GOOD MATCH
9,10,Abdelrahman Mohamed Elkady,Abdelrahman Mohamed Elkady.pdf,75.75,61.43,GOOD MATCH



REJECTED CANDIDATES


,rank,candidate_name,filename,ai_relevance_percent,final_match_percent,decision
63,64,Omar Ahmed Abdelraouf,Omar Ahmed Abdelraouf.pdf,18.0,0.0,REJECT
64,65,Alaa Karam Ahmed Mohamed Salama,Alaa Karam Ahmed Mohamed Salama.pdf,15.0,0.0,REJECT
65,66,Habiba Tarek Maher,Habiba Tarek Maher.pdf,15.0,0.0,REJECT
66,67,Mohamed Hisham Wafa,Mohamed Hisham Wafa.docx,15.0,0.0,REJECT
67,68,Ahmed Maged Mounir Mohammed,Ahmed Maged Mounir Mohammed.pdf,12.0,0.0,REJECT
68,69,Ather Walid Abdel Wahed Abou Shama,Ather Walid Abdel Wahed Abou Shama.pdf,12.0,0.0,REJECT
69,70,Faten Shahir Shokry Negeida,Faten Shahir Shokry Negeida.pdf,12.0,0.0,REJECT
70,71,Mirna Hany,Mirna Hany.pdf,12.0,0.0,REJECT
71,72,Mohamed Mokhtar Shaker,Mohamed Mokhtar Shaker.pdf,12.0,0.0,REJECT
72,73,Natalia Ehab Habib,Natalia Ehab Habib.pdf,12.0,0.0,REJECT


## 10. Mass Communication Test

The following unrelated CV should fail the AI relevance gate and receive approximately 0% final match.

In [17]:
mass_communication_cv = """
Mass Communication student with experience in social media marketing,
content creation, public relations, advertising campaigns, and brand communication.
"""

test_ai_relevance = calculate_ai_relevance(mass_communication_cv)

test_rows = []
for req_key, req in JOB_REQUIREMENTS.items():
    score, matched = calculate_keyword_score(mass_communication_cv, req)
    test_rows.append({
        "Requirement": req["name"],
        "Keyword Score": round(score * 100, 2),
        "Matched Terms": ", ".join(matched) if matched else "Not Found"
    })

print("AI Relevance:", round(test_ai_relevance * 100, 2), "%")
display(pd.DataFrame(test_rows))

if test_ai_relevance < AI_RELEVANCE_THRESHOLD:
    print("FINAL MATCH: 0%")
    print("DECISION: REJECT")
else:
    print("Review the AI relevance signals: the synthetic test passed the gate.")

AI Relevance: 0.0 %


,Requirement,Keyword Score,Matched Terms
0,ML Fundamentals,0.0,Not Found
1,Model Training / Evaluation / Optimization,0.0,Not Found
2,Hands-on AI Projects,0.0,Not Found
3,Backend Development,0.0,Not Found
4,REST APIs / FastAPI / Flask,0.0,Not Found
5,AI Model Integration / Deployment,0.0,Not Found


FINAL MATCH: 0%
DECISION: REJECT


## 11. Export Excel Report

In [18]:
output_path = "/content/AI_CV_Matching_Results.xlsx"

ranking_columns = [
    "rank", "candidate_name", "filename", "file_type",
    "processing_status", "ai_relevance_percent",
    "keyword_percent", "semantic_percent", "evidence_percent",
    "final_match_percent", "decision",
    "matched_skills", "missing_skills", "strong_evidence"
]

skill_columns = [
    col for col in results_df.columns
    if " - Keyword" in col or " - Semantic" in col or " - Evidence" in col
]

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    results_df[ranking_columns].to_excel(
        writer, sheet_name="Ranking", index=False
    )
    results_df[["rank", "candidate_name", "filename"] + skill_columns].to_excel(
        writer, sheet_name="Skill Analysis", index=False
    )
    rejected_df[ranking_columns].to_excel(
        writer, sheet_name="Rejected Candidates", index=False
    )
    results_df.head(TOP_N)[ranking_columns].to_excel(
        writer, sheet_name="Top Candidates", index=False
    )

print(f"Created: {output_path}")

Created: /content/AI_CV_Matching_Results.xlsx


## 12. Download Results

In [19]:
from google.colab import files

files.download("/content/AI_CV_Matching_Results.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>